# Candidate Enumeration
Configuration: `stages/02_candidates/config/runtime.json`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
candidate_enumeration_experiment_dir = ROOT / 'experiment/research/trapping_sets/stages/02_candidates'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

candidate_enumeration_data_paths = ExperimentPaths(ROOT, candidate_enumeration_experiment_dir / 'config/paths.json')

import json

candidate_enumeration_config = json.loads((candidate_enumeration_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.

from tqdm.auto import tqdm
from experiment.utils.trapping_sets import save_plotly, save_matplotlib

candidate_enumeration_plots = candidate_enumeration_data_paths.plots()
candidate_enumeration_plots.mkdir(parents=True, exist_ok=True)


In [ ]:
from experiment.utils import misc

def candidate_enumeration_aggregate_path(path):
    return misc.aggregate_path(path, data_paths=candidate_enumeration_data_paths)


# Trapping Set Analysis

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import numpy as np
import pandas as pd
import networkx as nx
from utils.graph_struct import build_tanner_graph_from_dem, induced_subgraph
from utils.graph_vis import visualize_det_on_graph_plotly, visualize_graph_plotly
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders.cpp_backend import BPOSD as candidate_enumeration_bposd

%load_ext autoreload
%autoreload 2


In [ ]:
candidate_enumeration_ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}
candidate_enumeration_labels = {
    '00': 'bf',
    '11': 'bs',
    '01': 'bfts',
    '10': 'bstf',
}


In [ ]:
from utils.misc import id2str, str2id as candidate_enumeration_str2id
from utils.misc import pack_bits_little, unpack_bits_little

def construct_fault_in_weight(candidate, num_vars, weight):
    faults = []
    for var_ids in tqdm(combinations(candidate, weight), desc='Trapping-set analysis', leave=True, disable=candidate_enumeration_config['print_level'] == 1):
        fault = np.zeros(num_vars, dtype=np.uint8)
        fault[list(var_ids)] = 1
        faults.append(fault)

    return np.array(faults)

def trapping_set_candidates(G: nx.Graph, a_max: int, b_max: int):
    """
    Finds candidate trapping sets in the graph G with parameters (a, b) s.t. a <= a_max and b <= b_max.

    (a, b) classical-type TS: a variable nodes, b odd-degree check nodes
    (a, 0) symmetric stabilizer TS: a variable nodes, 0 odd-degree check nodes, and a is even.

    Usually, we are interested in small trapping sets (small a and b) since they are more likely to cause decoding failures.

    Parameters
    ----------
    G: nx.Graph
        the tanner graph built from the DEM.
    a_max: int
        maximum number of variable nodes in the trapping set.
    b_max: int
        maximum number of odd-degree check nodes in the trapping set.

    Returns
    ----------
    ct_candidates: dict
        dictionary mapping (a, b) to list of candidate classical-type trapping sets, where each candidate is represented as a set of node ids (including only variable nodes).
    ss_candidates: dict
        dictionary mapping (a, b) to list of candidate symmetric stabilizer trapping sets, where each candidate is represented as a set of node ids (including only variable nodes).
    """

    ct_candidates = defaultdict(list)
    ss_candidates = defaultdict(list)
    error_nodes = [n for n, d in G.nodes(data=True) if d['kind'] == 'error']
    detector_nodes = [n for n, d in G.nodes(data=True) if d['kind'] == 'detector']
    num_dets = len(detector_nodes)
    a_bound = min(a_max, len(error_nodes))

    for a in tqdm(range(1, a_bound + 1), desc='Trapping-set analysis', leave=True, disable=candidate_enumeration_config['print_level'] == 1):
        for es in tqdm(combinations(error_nodes, a), desc='Trapping-set analysis', leave=False, disable=candidate_enumeration_config['print_level'] == 1):
            neighbor_checks = set()
            for e in es:
                neighbor_checks.update(G.neighbors(e))
            SG = G.subgraph(neighbor_checks.union(es))
            odd_degree_checks = [c for c in neighbor_checks if SG.degree(c) % 2 == 1]
            b = len(odd_degree_checks)

            if b <= b_max:
                ct_candidates[a, b].append(set(map(lambda x: x - num_dets, es)))
            if b == 0 and a % 2 == 0:
                ss_candidates[a, b].append(set(map(lambda x: x - num_dets, es)))

    return (ct_candidates, ss_candidates)

def candidates_to_fault_det(
    ct_candidates,
    ss_candidates,
    H_dem,
    num_vars,
    weight,
):
    """
    Converts the candidate trapping sets to their corresponding faults (and then detector events by applying the detector error matrix).

    Parameters
    ----------
    ct_candidates: dict
        dictionary mapping (a, b) to list of candidate classical-type trapping sets, where each candidate is represented as a set of node ids (including only variable nodes).
    ss_candidates: dict
        dictionary mapping (a, b) to list of candidate symmetric stabilizer trapping sets, where each candidate is represented as a set of node ids (including only variable nodes).
    H_dem: np.ndarray
        the detector error matrix.
    num_vars: int
        the number of variable nodes.
    weight: int
        the weight of faults to consider.

    Returns
    ----------
    ct_faults: dict
    ss_faults: dict
    ct_dets: dict
    ss_dets: dict
    """

    ct_faults, ss_faults = ({}, {})
    ct_dets, ss_dets = ({}, {})
    if len(ct_candidates) > 0:
        for (a, b), candidates in tqdm(ct_candidates.items(), desc='Trapping-set analysis', leave=True, disable=candidate_enumeration_config['print_level'] == 1):
            if a < weight:
                continue
            faults = np.vstack([construct_fault_in_weight(candidate, num_vars, weight) for candidate in candidates], dtype=np.uint8)
            unique_faults = np.unique(faults, axis=0)
            ct_faults[a, b] = unique_faults
            dets = unique_faults @ H_dem.T % 2
            ct_dets[a, b] = dets

    if len(ss_candidates) > 0:
        for (a, b), candidates in tqdm(ss_candidates.items(), desc='Trapping-set analysis', leave=True, disable=candidate_enumeration_config['print_level'] == 1):
            if a < weight:
                continue
            faults = np.vstack([construct_fault_in_weight(candidate, num_vars, weight) for candidate in candidates], dtype=np.uint8)
            unique_faults = np.unique(faults, axis=0)
            ss_faults[a, b] = unique_faults
            dets = unique_faults @ H_dem.T % 2
            ss_dets[a, b] = dets

    return (ct_faults, ss_faults, ct_dets, ss_dets)

def decode_on_candidates(
    dem,
    compiled_decoder,
    ct_candidates,
    ss_candidates,
    num_vars,
    save_folder: Path,
):
    from beliefmatching import detector_error_model_to_check_matrices

    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H_dem = demMatrices.check_matrix
    L_dem = demMatrices.observables_matrix
    num_obs = L_dem.shape[0]
    overview = []
    failure_cases = []
    max_weight = max(max((a for a, b in ct_candidates.keys()), default=0), max((a for a, b in ss_candidates.keys()), default=0))

    for weight in tqdm(range(1, max_weight + 1), desc='Trapping-set analysis', leave=True, disable=candidate_enumeration_config['print_level'] == 1):
        if candidate_enumeration_config['print_level'] >= 2:
            print(f'Analyzing candidates with fault weight {weight}...')
        ct_faults, ss_faults, ct_dets, ss_dets = candidates_to_fault_det(ct_candidates, ss_candidates, H_dem, num_vars, weight)

        for (a, b), dets in tqdm(ct_dets.items(), desc='Trapping-set analysis', leave=False, disable=candidate_enumeration_config['print_level'] == 1):
            faults = ct_faults[a, b]
            obs = faults @ L_dem.T % 2
            dets_packed = pack_bits_little(dets)
            pred_packed = compiled_decoder.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
            pred = unpack_bits_little(pred_packed, num_obs)
            succ = np.all(pred == obs, axis=1)

            if candidate_enumeration_config['print_level'] >= 2:
                print(f'    Classical-type TS (a={a}, b={b}): total {len(succ)}, {np.sum(succ)} succeeded, {len(succ) - np.sum(succ)} failed, success rate {np.sum(succ) / len(succ) * 100:.2f}%')
            overview.append({
                'type': 'classical-type',
                'a': a,
                'b': b,
                'weight': weight,
                'total': len(succ),
                'succeeded': np.sum(succ),
                'failed': len(succ) - np.sum(succ),
            })
            failure_cases.extend([{
                'type': 'classical-type',
                'a': a,
                'b': b,
                'weight': weight,
                'fault': ''.join(map(str, faults[i])),
                'det': ''.join(map(str, dets[i])),
                'obs': ''.join(map(str, obs[i])),
                'pred': ''.join(map(str, pred[i])),
            } for i in range(len(succ)) if not succ[i]])

        if candidate_enumeration_config['print_level'] >= 2:
            print('---------------------------------------------------------------------------')
        for (a, b), dets in tqdm(ss_dets.items(), desc='Trapping-set analysis', leave=False, disable=candidate_enumeration_config['print_level'] == 1):
            faults = ss_faults[a, b]
            obs = faults @ L_dem.T % 2
            dets_packed = pack_bits_little(dets)
            pred_packed = compiled_decoder.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
            pred = unpack_bits_little(pred_packed, num_obs)
            succ = np.all(pred == obs, axis=1)

            if candidate_enumeration_config['print_level'] >= 2:
                print(f'    Symmetric stabilizer TS (a={a}, b={b}): total {len(succ)}, {np.sum(succ)} succeeded, {len(succ) - np.sum(succ)} failed, success rate {np.sum(succ) / len(succ) * 100:.2f}%')
            overview.append({
                'type': 'symmetric stabilizer',
                'a': a,
                'b': b,
                'weight': weight,
                'total': len(succ),
                'succeeded': np.sum(succ),
                'failed': len(succ) - np.sum(succ),
            })
            failure_cases.extend([{
                'type': 'symmetric stabilizer',
                'a': a,
                'b': b,
                'weight': weight,
                'fault': ''.join(map(str, faults[i])),
                'det': ''.join(map(str, dets[i])),
                'obs': ''.join(map(str, obs[i])),
                'pred': ''.join(map(str, pred[i])),
            } for i in range(len(succ)) if not succ[i]])

        if candidate_enumeration_config['print_level'] >= 2:
            print('===========================================================================')

    overview, failure_cases = (pd.DataFrame(overview), pd.DataFrame(failure_cases))
    overview.to_csv(candidate_enumeration_aggregate_path(save_folder / 'TS_overview.csv'), index=False)
    failure_cases.to_feather(save_folder / 'TS_failure_cases.feather')

    return (overview, failure_cases)


### Load and reconstruct a failure example

Read the with-errors case files and rebuild the selected example's DEM/Tanner graph. Its code and p determine the context for candidate enumeration and decoding.


In [ ]:
candidate_enumeration_data_dir = candidate_enumeration_data_paths.output('circuit_level')
candidate_enumeration_cases_dir = candidate_enumeration_data_paths.input('failure_cases_with_errors')
assert candidate_enumeration_cases_dir.is_dir(), candidate_enumeration_cases_dir


In [ ]:
candidate_enumeration_bfts_cases_path = candidate_enumeration_data_paths.input('ca_bfts_cases_csv')
candidate_enumeration_bstf_cases_path = candidate_enumeration_data_paths.input('ca_bstf_cases_csv')
candidate_enumeration_bs_cases_path = candidate_enumeration_data_paths.input('ca_bs_cases_csv')
candidate_enumeration_bf_cases_path = candidate_enumeration_data_paths.input('ca_bf_cases_csv')
candidate_enumeration_types = {
    'code': str,
    'p': float,
    'detector_events': str,
    'error_events': str,
    'label': str,
    'true_observables': str,
    'bposd_pred_observables': str,
    'tesseract_pred_observables': str,
}

candidate_enumeration_bfts_cases = pd.read_csv(candidate_enumeration_bfts_cases_path, dtype=candidate_enumeration_types)
candidate_enumeration_bstf_cases = pd.read_csv(candidate_enumeration_bstf_cases_path, dtype=candidate_enumeration_types)
candidate_enumeration_bs_cases = pd.read_csv(candidate_enumeration_bs_cases_path, dtype=candidate_enumeration_types)
candidate_enumeration_bf_cases = pd.read_csv(candidate_enumeration_bf_cases_path, dtype=candidate_enumeration_types)

candidate_enumeration_bfts_cases['label'] = '01'
candidate_enumeration_bstf_cases['label'] = '10'
candidate_enumeration_bs_cases['label'] = '11'
candidate_enumeration_bf_cases['label'] = '00'
candidate_enumeration_bfts_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
candidate_enumeration_bstf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
candidate_enumeration_bs_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
candidate_enumeration_bf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)


In [ ]:
candidate_enumeration_bfts_cases


In [ ]:
candidate_enumeration_row = candidate_enumeration_bfts_cases.iloc[-1]
candidate_enumeration_code = candidate_enumeration_ca_codes[candidate_enumeration_row['code']]
candidate_enumeration_p = float(candidate_enumeration_row['p'])
candidate_enumeration_builder = CircuitBuilder(candidate_enumeration_code)
candidate_enumeration_circ = candidate_enumeration_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=candidate_enumeration_p, p_meas=0.0), d_rounds=1)
candidate_enumeration_dem = candidate_enumeration_circ.detector_error_model()
candidate_enumeration_compiled_decoder = candidate_enumeration_bposd(candidate_enumeration_code, candidate_enumeration_p).compile_decoder_for_dem(dem=candidate_enumeration_dem)
candidate_enumeration_g, candidate_enumeration_ = build_tanner_graph_from_dem(candidate_enumeration_dem)
num_vars = sum((1 for n, d in candidate_enumeration_g.nodes(data=True) if d['kind'] == 'error'))
num_dets = sum((1 for n, d in candidate_enumeration_g.nodes(data=True) if d['kind'] == 'detector'))


### Enumerate and decode candidate supports

Set the candidate-size bounds, construct classical-type and symmetric-stabilizer candidates, then test their decoding outcomes. A candidate definition is not a guarantee that every candidate fails.


In [ ]:
a_max, b_max = (6, 10)
save_folder = candidate_enumeration_data_paths.output(
    'parameter_1_p_parameter_2_a_max_a_max_b_max_b_max',
    parameter_1=candidate_enumeration_row['code'],
    parameter_2=candidate_enumeration_row['p'],
    a_max=a_max,
    b_max=b_max,
)
save_folder.mkdir(parents=True, exist_ok=True)


In [ ]:
ct_candidates, ss_candidates = trapping_set_candidates(candidate_enumeration_g, a_max=a_max, b_max=b_max)


In [ ]:
overview, failure_cases = decode_on_candidates(candidate_enumeration_dem, candidate_enumeration_compiled_decoder, ct_candidates, ss_candidates, num_vars, save_folder=save_folder)


### Summarize the candidate outcomes

Reload the aggregate overview and compare success/failure counts by weight. These fractions describe the enumerated candidates under the configured decoder, not physical-noise LER.


In [ ]:
overview = pd.read_csv(candidate_enumeration_aggregate_path(candidate_enumeration_data_paths.output(
    'TS_overview_csv',
    parameter_1=candidate_enumeration_row['code'],
    parameter_2=candidate_enumeration_row['p'],
    a_max=a_max,
    b_max=b_max,
)))
failure_cases = pd.read_feather(candidate_enumeration_data_paths.output(
    'TS_failure_cases_feather',
    parameter_1=candidate_enumeration_row['code'],
    parameter_2=candidate_enumeration_row['p'],
    a_max=a_max,
    b_max=b_max,
))


In [ ]:
overview[overview['failed'] > 0].sort_values(by='weight')


In [ ]:
error_weight_dist = overview.groupby('weight')[['total', 'succeeded', 'failed']].sum()
error_weight_dist['succeeded_rate'] = error_weight_dist['succeeded'] / error_weight_dist['total']
error_weight_dist['failed_rate'] = error_weight_dist['failed'] / error_weight_dist['total']
candidate_enumeration_stats = candidate_enumeration_data_paths.stats()
candidate_enumeration_stats.mkdir(parents=True, exist_ok=True)

error_weight_dist.to_csv(candidate_enumeration_data_paths.output('candidate_error_weight_distribution_csv'))
error_weight_dist


In [ ]:
save_matplotlib(error_weight_dist.plot(kind='bar', y=['succeeded_rate', 'failed_rate'], stacked=True), candidate_enumeration_data_paths.output('cell_0016_1_png'))


### Inspect failures by candidate family

Keep classical-type and symmetric-stabilizer cases separate while plotting their detector/error supports. Representative failures illustrate the mechanism hypothesis but do not establish behavior for every embedding.


In [ ]:
ct_fail_cases = failure_cases[failure_cases['type'] == 'classical-type']
ct_fail_cases.tail(10)


In [ ]:
ss_fail_cases = failure_cases[failure_cases['type'] == 'symmetric stabilizer']
ss_fail_cases[ss_fail_cases['a'] == ss_fail_cases['weight']]


In [ ]:
save_plotly(visualize_det_on_graph_plotly(candidate_enumeration_g, 'tanner', ct_fail_cases.iloc[-1]['det'], other_events={('fault', 'orange', 'circle'): ct_fail_cases.iloc[-1]['fault']}), candidate_enumeration_data_paths.output('cell_0019_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(induced_subgraph(candidate_enumeration_g, candidate_enumeration_str2id(ct_fail_cases.iloc[-1]['fault'], offset=num_dets))), candidate_enumeration_data_paths.output('cell_0020_1_html'))


In [ ]:
save_plotly(visualize_det_on_graph_plotly(candidate_enumeration_g, 'tanner', ss_fail_cases.iloc[-1]['det'], other_events={('fault', 'orange', 'circle'): ss_fail_cases.iloc[-1]['fault']}), candidate_enumeration_data_paths.output('cell_0021_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(induced_subgraph(candidate_enumeration_g, candidate_enumeration_str2id(ss_fail_cases.iloc[-1]['fault'], offset=num_dets))), candidate_enumeration_data_paths.output('cell_0022_1_html'))


In [ ]:
save_plotly(visualize_det_on_graph_plotly(candidate_enumeration_g, 'tanner', ss_fail_cases.iloc[-3]['det'], other_events={('fault', 'orange', 'circle'): ss_fail_cases.iloc[-3]['fault']}), candidate_enumeration_data_paths.output('cell_0023_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(induced_subgraph(candidate_enumeration_g, candidate_enumeration_str2id(ss_fail_cases.iloc[-3]['fault'], offset=num_dets))), candidate_enumeration_data_paths.output('cell_0024_1_html'))


In [ ]:
save_plotly(visualize_det_on_graph_plotly(candidate_enumeration_g, 'tanner', ss_fail_cases.iloc[0]['det'], other_events={('fault', 'orange', 'circle'): ss_fail_cases.iloc[0]['fault']}), candidate_enumeration_data_paths.output('cell_0025_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(induced_subgraph(candidate_enumeration_g, candidate_enumeration_str2id(ss_fail_cases.iloc[0]['fault'], offset=num_dets))), candidate_enumeration_data_paths.output('cell_0026_1_html'))


### Inspect a boundary-parameter subset

Filter the recorded candidates by their trapping parameter and display the selected example. Keep the selected support and decoding context with the graph interpretation.


In [ ]:
failure_cases[failure_cases['b'] == 10]


In [ ]:
save_plotly(visualize_det_on_graph_plotly(
    candidate_enumeration_g,
    'tanner',
    failure_cases[failure_cases['b'] == 10].iloc[0]['det'],
    other_events={
        ('fault', 'orange', 'circle'): failure_cases[failure_cases['b'] == 10].iloc[0]['fault'],
    },
), candidate_enumeration_data_paths.output('cell_0028_1_html'))
